# Simulación HetNet paso a paso

Este notebook reproduce el flujo de `main.py` para entender cada etapa: generación de usuarios, ejecución de escenarios, métricas y visualización.

## 1) Importaciones y configuración inicial

Cargamos dependencias, añadimos el proyecto al `PYTHONPATH` y definimos parámetros base.

In [ ]:
import os
import sys
import copy
import numpy as np
import matplotlib.pyplot as plt

# Asegura imports del proyecto cuando se ejecuta desde el notebook
PROJECT_ROOT = os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.network import MacroCell, SmallCell, UserEquipment, Network
from src.scheduler import ProportionalFairScheduler
from src.analysis import (
    plot_rsrp_map,
    plot_deployment_map,
    plot_throughput_comparison,
    plot_sinr_histogram,
    plot_load_distribution,
    handover_analysis,
    RESULTS_DIR,
)

AREA_M = (1000, 1000)
MACRO_X, MACRO_Y = 500, 500
SC1_X, SC1_Y = 700, 650
SC2_X, SC2_Y = 350, 300

USERS = 50
SEED = 42
CRE_DB = 0.0
PF_STEPS = 200
HOTSPOT = True

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results dir: {RESULTS_DIR}')

## 2) Funciones auxiliares (igual que en el principal)

Definimos estaciones base, generación de usuarios y ejecución RR/PF.

In [ ]:
def build_base_stations(include_small_cells=True):
    macro = MacroCell(
        x=MACRO_X, y=MACRO_Y, tx_power_dbm=45.0, height_m=25.0,
        freq_mhz=1800.0, bandwidth_mhz=20.0, name='Macro-1800'
    )
    if not include_small_cells:
        return [macro]

    sc1 = SmallCell(
        x=SC1_X, y=SC1_Y, tx_power_dbm=27.0, height_m=5.0,
        freq_mhz=2600.0, bandwidth_mhz=20.0, name='SC1-2600'
    )
    sc2 = SmallCell(
        x=SC2_X, y=SC2_Y, tx_power_dbm=27.0, height_m=5.0,
        freq_mhz=2600.0, bandwidth_mhz=20.0, name='SC2-2600'
    )
    return [macro, sc1, sc2]


def generate_users(n_users, seed=42, hotspot=True):
    rng = np.random.default_rng(seed)
    users = []
    uid = 0

    if hotspot:
        n_uniform = int(n_users * 0.40)
        n_hot1 = int(n_users * 0.30)
        n_hot2 = n_users - n_uniform - n_hot1

        xs = rng.uniform(0, AREA_M[0], n_uniform)
        ys = rng.uniform(0, AREA_M[1], n_uniform)
        for x, y in zip(xs, ys):
            users.append(UserEquipment(x=float(x), y=float(y), uid=uid))
            uid += 1

        xs = np.clip(rng.normal(SC1_X, 80, n_hot1), 0, AREA_M[0])
        ys = np.clip(rng.normal(SC1_Y, 80, n_hot1), 0, AREA_M[1])
        for x, y in zip(xs, ys):
            users.append(UserEquipment(x=float(x), y=float(y), uid=uid))
            uid += 1

        xs = np.clip(rng.normal(SC2_X, 80, n_hot2), 0, AREA_M[0])
        ys = np.clip(rng.normal(SC2_Y, 80, n_hot2), 0, AREA_M[1])
        for x, y in zip(xs, ys):
            users.append(UserEquipment(x=float(x), y=float(y), uid=uid))
            uid += 1
    else:
        xs = rng.uniform(0, AREA_M[0], n_users)
        ys = rng.uniform(0, AREA_M[1], n_users)
        for x, y in zip(xs, ys):
            users.append(UserEquipment(x=float(x), y=float(y), uid=uid))
            uid += 1

    return users


def run_scenario_rr(base_stations, users, cre_offset_db=0.0):
    net = Network(base_stations, users, cre_offset_db=cre_offset_db)
    net.run()
    return net


def run_scenario_pf(base_stations, users, cre_offset_db=0.0, pf_steps=200):
    net = Network(base_stations, users, cre_offset_db=cre_offset_db)
    net.assign_cells()
    sched = ProportionalFairScheduler(net, alpha=0.9)
    sched.run(n_steps=pf_steps)
    return net


def print_report(label, net, ho_stats=None):
    print('=' * 60)
    print(f'Escenario: {label}')
    print('=' * 60)
    print(f'Usuarios totales  : {len(net.users)}')
    print(f'Throughput medio  : {net.mean_throughput_bps() / 1e6:.2f} Mbps')
    print(f'SINR media        : {net.mean_sinr_db():.1f} dB')
    edge = net.edge_users()
    print(f'Usuarios de borde : {len(edge)} ({100 * len(edge) / max(len(net.users), 1):.1f} %)')
    print('Carga por celda:')
    for bs_name, count in net.users_per_bs().items():
        print(f'  - {bs_name}: {count}')

    if ho_stats:
        print('--- Handover ---')
        print(f"Handovers activados : {ho_stats['handover_triggered']} ({ho_stats['handover_pct']:.1f} %)")
        print(f"Offload de macro    : {ho_stats['macro_offload_pct']:.1f} %")

## 3) Generar usuarios y preparar escenarios

In [ ]:
users_base = generate_users(USERS, seed=SEED, hotspot=HOTSPOT)
bs_macro = build_base_stations(include_small_cells=False)
bs_hetnet = build_base_stations(include_small_cells=True)

print(f'Usuarios generados: {len(users_base)}')
print(f'BS macro-only: {len(bs_macro)} | BS hetnet: {len(bs_hetnet)}')

## 4) Ejecutar escenarios (RR y PF)

In [ ]:
net_macro = run_scenario_rr(bs_macro, [copy.deepcopy(u) for u in users_base])
net_hetnet = run_scenario_rr(bs_hetnet, [copy.deepcopy(u) for u in users_base], cre_offset_db=CRE_DB)
net_hetnet_pf = run_scenario_pf(build_base_stations(True), [copy.deepcopy(u) for u in users_base], cre_offset_db=CRE_DB, pf_steps=PF_STEPS)

ho_stats = handover_analysis(net_macro, net_hetnet)

print_report('Solo Macro (RR)', net_macro)
print()
print_report('Macro + Small Cells (RR)', net_hetnet, ho_stats)
print()
print_report('Macro + Small Cells (PF)', net_hetnet_pf)

## 5) Generar gráficas

In [ ]:
p1 = plot_rsrp_map(bs_macro, area_m=AREA_M, title='RSRP – Solo Macrocelda', filename='rsrp_macro_only.png')
p2 = plot_rsrp_map(bs_hetnet, area_m=AREA_M, title='RSRP – Macro + Small Cells', filename='rsrp_hetnet.png')
p3 = plot_deployment_map(bs_macro, net_macro.users, area_m=AREA_M, title='Emplazamientos – Solo Macrocelda', filename='deployment_macro_only.png')
p4 = plot_deployment_map(bs_hetnet, net_hetnet.users, area_m=AREA_M, title='Emplazamientos – Macro + Small Cells', filename='deployment_hetnet.png')

scenarios = [
    {'label': 'Solo Macro (RR)', 'users': net_macro.users},
    {'label': 'HetNet RR', 'users': net_hetnet.users},
    {'label': 'HetNet PF', 'users': net_hetnet_pf.users},
]

p5 = plot_throughput_comparison(scenarios, filename='throughput_comparison.png')
p6 = plot_sinr_histogram(scenarios, filename='sinr_histogram.png')
p7 = plot_load_distribution(net_hetnet, title='Distribución de carga – Macro + Small Cells (RR)', filename='load_distribution.png')

generated = [p1, p2, p3, p4, p5, p6, p7]
print('Ficheros generados:')
for path in generated:
    print(' -', os.path.basename(path))

## 6) Visualización rápida dentro del notebook

Mostramos todas las figuras en una rejilla para análisis rápido.

In [ ]:
images = [
    'rsrp_macro_only.png',
    'rsrp_hetnet.png',
    'deployment_macro_only.png',
    'deployment_hetnet.png',
    'throughput_comparison.png',
    'sinr_histogram.png',
    'load_distribution.png',
]

fig, axes = plt.subplots(4, 2, figsize=(14, 20))
axes = axes.flatten()

for i, name in enumerate(images):
    path = os.path.join(RESULTS_DIR, name)
    if os.path.exists(path):
        img = plt.imread(path)
        axes[i].imshow(img)
        axes[i].set_title(name)
    else:
        axes[i].text(0.5, 0.5, f'No encontrado: {name}', ha='center', va='center')
    axes[i].axis('off')

for j in range(len(images), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

## 7) Próximos experimentos

- Cambiar `USERS` a 100 y observar throughput.
- Probar `CRE_DB = 6` para ampliar cobertura de small cells.
- Activar/desactivar `HOTSPOT` para comparar distribución uniforme vs concentrada.